In [1]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import DataLoader
from torch.utils.data import random_split # use for data distribution for clients

In [2]:
# Data Loading
transform = transforms.ToTensor()
mnist_trainset=datasets.MNIST(root = './data', train = True, download = True, transform = transform)
mnist_testset=datasets.MNIST(root = './data', train = False, download = True, transform = transform)
image, label = mnist_trainset[0]

In [3]:
# CNN
# a CNN with 2 layers: Layer1-> Relu->Pool->Layer2->...->Flatten->Fully connected->Output
class CNN(nn.Module): # we take the standard class and modify it
    def __init__(self): # this is a constructor
        super(CNN,self).__init__()
        # First layer
        self.conv1 = nn.Conv2d(in_channels=1,   # Mnist has 1 channel
                          out_channels=16, # 16 filters
                          kernel_size=3)   # 3x3 kernel size
        # Second layer
        self.conv2 = nn.Conv2d(in_channels=16,   # Mnist has 1 channel
                          out_channels=32, # 16 filters
                          kernel_size=3)   # 3x3 kernel size
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)
        # create the fully connected layer
        self.fully_connected = nn.Linear(800,10)

    def forward(self,x):
        x = self.pool(self.relu(self.conv1(x))) # do relu and pooling for layer 1
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # here we reshape the tensor to be of 'out.size(0)' rows 
                                  # but don't know exactly how many columns, so we specify '-1'
                                  # [1, 32, 5, 5] -> [1, 800]
        x = self.fully_connected(x)
        return x

model = CNN()
out = model(image.unsqueeze(0))
print(out.shape)

torch.Size([1, 10])


In [4]:
# Client abstraction, split dataset into clients
number_of_clients = 5

dataset_size = len(mnist_trainset)
client_size = dataset_size // number_of_clients # we use // instead of / to remove the fractional part
client_datasets = random_split(mnist_trainset, [client_size]*number_of_clients) # a list with: client_datasets[0], client_datasets[1], ...[5]

client_loaders = []
for dataset in client_datasets:
    loader = DataLoader(dataset, batch_size = 64, shuffle = True)
    client_loaders.append(loader)

print(len(client_loaders))
print(len(client_loaders[0]))
images, labels = next(iter(client_loaders[0]))
print(images.shape)

5
188
torch.Size([64, 1, 28, 28])


In [5]:
# Do training locally
def train_local(model, dataloader, epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimiser = optim.SGD(model.parameters(), lr = lr)
    
    model.train()
    for epoch in range (epochs):
        running_loss = 0.0
        for images, labels in dataloader:
            # forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            # backward pass
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
    
            running_loss += loss.item()
        avg_loss = running_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")

    return model.state_dict() # return a dictionary of model parameters

# client_model = CNN()
# weights = train_local(client_model, client_loaders[0], epochs = 1, lr = 0.1)
# print(weights.keys())

In [6]:
# Now we can do it for multiple clients
global_model = CNN()
# weights = train_local(local_model, client_loaders[0], epochs = 3, lr = 0.1)
client_state_dictionary = []
number_of_samples = []
# loop over all clients
for client in range(number_of_clients):
    local_model = copy.deepcopy(global_model) # copy to local model
    weights = train_local(local_model, client_loaders[client], epochs=1, lr=0.1) # train it
    client_state_dictionary.append(weights) # store the weights
    number_of_samples.append(len(client_loaders[client].dataset)) # store number of samples
print(len(client_state_dictionary))
print(number_of_samples)
print(client_state_dictionary[0].keys())

Epoch 1/1, Average Loss: 0.7178
Epoch 1/1, Average Loss: 0.7310
Epoch 1/1, Average Loss: 0.7109
Epoch 1/1, Average Loss: 0.7204
Epoch 1/1, Average Loss: 0.7088
5
[12000, 12000, 12000, 12000, 12000]
odict_keys(['conv1.weight', 'conv1.bias', 'conv2.weight', 'conv2.bias', 'fully_connected.weight', 'fully_connected.bias'])


In [19]:
# Now we need to aggregate the results with FedAvg
global_state_dictionary = {} # here averaged weights will be stored
for key in client_state_dictionary[0].keys(): # here client_state_dictionary[0].keys() is ok because all clients have the same keys:
                                              # ['conv1.weight', 'conv1.bias', 'conv2.weight', 'conv2.bias', 'fully_connected.weight', 'fully_connected.bias']
    global_state_dictionary[key] = torch.zeros_like(client_state_dictionary[0][key]) # initialise with first client

for client_id, state_dictionary in enumerate(client_state_dictionary): # iterate over clients
    weight = number_of_samples[client_id] / total_samples # we normalise because different clients can have different data size
    for key in state_dictionary.keys(): # iterate over keys of each client
        global_state_dictionary[key] += state_dictionary[key] * weight # eg. global_layer = sum(client_layer * weight)
# set global model parameters for the next step
global_model.load_state_dict(global_state_dictionary)



NameError: name 'client_state_dictionary' is not defined